<div style="border-radius:10px;overflow:hidden;font-family:Aptos,Calibri,Segoe UI,sans-serif;border:1px solid #e6e9ee">
  <div style="background:#0A2540;padding:18px 22px;color:#fff">
    <div style="font-size:12px;letter-spacing:2px;color:#F4B942;font-weight:700">CARISURG · MEDTECH PATHWAYS · WEEK 5 · TUTORIAL 1</div>
    <div style="font-size:24px;font-weight:700;margin-top:4px">Clinical Data Literacy</div>
    <div style="font-size:14px;color:#cdd6df;margin-top:6px">Reading a real ED triage dataset like a clinician — the meaning behind every column, before any modelling.</div></div>
  <div style="background:#1B9AAA;color:#FFFFFF;padding:6px 22px;font-size:12px;font-weight:700;letter-spacing:1px">INSTRUCTOR NOTEBOOK</div>
</div>

## Why this session exists

Mercer General's ED Board has handed your team a de-identified arrivals dataset — the **Yale EMMLC triage extract** — and one question: *before we build anything, do we actually understand what each column means?*

Today is **literacy — not modelling, and not yet plots**. A model is only as trustworthy as the people who understand its inputs. So this session does one thing: it grounds you in the **clinical meaning of every feature category** — who the patient is, what was measured at the door, what they came in for, and what we are trying to predict.

**By the end you can:**
- name the feature families — identifiers & outcomes, arrival vitals, and ~200 chief-complaint flags — and say what each is *for* clinically;
- read the triage vitals and their normal adult ranges (and spot the Fahrenheit trap);
- explain what a chief complaint is, and how 200 binary flags collapse into a handful of body systems;
- explain **why `esi` (Triage_Level) is our target**, and what **ESI's five levels** mean.

We will read the file, print its columns, and check the *type* of every field — but we draw **no charts today**. Distributions and visuals are Tutorial 4.

### Where this sits in Week 5
`T1 Literacy →` **T2 Profiling** `→` **T3 Cleaning** `→` **T4 Visualisation** `→` **T5 Feasibility Memo**. The same dataset carries into Week 6's baseline model. Everything downstream depends on getting the *meaning* right today.

<div style="border-left:4px solid #C0392B;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#C0392B">⚠️ DATA / SAFETY NOTE</b><br>This is a de-identified <b>US academic-hospital</b> dataset — real identifiers (name, MRN, date of birth, address) have been removed. Keep it that way: paste only <b>column names, schemas, and summary statistics</b> into any AI tool — <b>never full patient rows</b>. And hold one question live all week: <i>how well does a US triage pattern transfer to a Caribbean ED?</i></div>

## 1 · Setup

In [1]:
# Run this first. For a pure-literacy session we need only two libraries —
# no plotting today (charts arrive in Tutorial 4).
import numpy as np    # numerical helpers (NaN, medians, ...)
import pandas as pd   # tables / DataFrames — our main tool all week

# Let pandas show more of a wide table when we print it:
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

print("Environment ready · pandas", pd.__version__)

Environment ready · pandas 2.2.2


## 2 · Load the data, then read its shape, columns, and types

With a wide dataset the first three questions are always: *how big is it, what are the columns, and what type is each one?*

In [3]:
from pathlib import Path
# Google Colab: mount Drive, then point DATA_PATH at the file (~55 MB, give it a few seconds):
#   from google.colab import drive; drive.mount("/content/drive")
#   DATA_PATH = "/content/drive/MyDrive/CariSurg/yaleemmlc_admissionprediction_triage.csv"
DATA_PATH = Path("yaleemmlc_admissionprediction_triage.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find {DATA_PATH}. Place the CSV beside this notebook, or set DATA_PATH (Colab).")

# index_col=0 drops the unnamed export row-number column (see the identifiers note below).
df = pd.read_csv(DATA_PATH, index_col=0)
print(f"Loaded {df.shape[0]:,} ED encounters  x  {df.shape[1]} columns")

Loaded 55,121 ED encounters  x  225 columns


In [4]:
df.head()

,dep_name,esi,age,gender,ethnicity,race,lang,religion,maritalstatus,employstatus,insurance_status,disposition,arrivalmode,arrivalmonth,arrivalday,arrivalhour_bin,previousdispo,triage_vital_hr,triage_vital_sbp,triage_vital_dbp,triage_vital_rr,triage_vital_o2,triage_vital_o2_device,triage_vital_temp,triage_glucose,cc_abdominalcramping,cc_abdominaldistention,cc_abdominalpain,cc_abdominalpainpregnant,cc_abnormallab,...,cc_sorethroat,cc_stdcheck,cc_strokealert,cc_suicidal,cc_suture/stapleremoval,cc_swallowedforeignbody,cc_syncope,cc_tachycardia,cc_testiclepain,cc_thumbinjury,cc_tickremoval,cc_toeinjury,cc_toepain,cc_trauma,cc_unresponsive,cc_uri,cc_urinaryfrequency,cc_urinaryretention,cc_urinarytractinfection,cc_vaginalbleeding,cc_vaginaldischarge,cc_vaginalpain,cc_weakness,cc_wheezing,cc_withdrawal-alcohol,cc_woundcheck,cc_woundinfection,cc_woundre-evaluation,cc_wristinjury,cc_wristpain
7,A,4.0,87.0,Female,Hispanic or Latino,Other,Other,Pentecostal,Widowed,Retired,Medicare,Discharge,Car,March,Saturday,11-14,Admit,88.0,155.0,75.0,17.0,98.0,0.0,97.8,87.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17,B,2.0,53.0,Male,Hispanic or Latino,Other,English,Catholic,Significant Other,Disabled,Medicare,Admit,Walk-in,September,Monday,11-14,Discharge,118.0,105.0,79.0,20.0,98.0,0.0,97.5,113.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
40,A,2.0,49.0,Female,Non-Hispanic,White or Caucasian,English,Catholic,Married,Full Time,Commercial,Discharge,ambulance,June,Tuesday,15-18,Admit,76.0,116.0,71.0,18.0,99.0,0.0,98.1,108.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
47,A,3.0,22.0,Female,Hispanic or Latino,White or Caucasian,English,Catholic,Single,Full Time,Commercial,Discharge,Car,April,Sunday,23-02,Discharge,106.0,103.0,63.0,16.0,97.0,0.0,98.2,85.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60,A,2.0,62.0,Male,Non-Hispanic,White or Caucasian,English,Protestant,Divorced,Not Employed,Medicaid,Discharge,ambulance,July,Wednesday,23-02,Discharge,84.0,109.0,68.0,18.0,95.0,0.0,97.8,88.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


<div style="border-left:4px solid #6C5CE7;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#6C5CE7">🧩 PANDAS IN PLAIN ENGLISH</b><br><code>pd.read_csv(...)</code> reads the file into a <b>DataFrame</b> — a table. <code>df.shape</code> is <code>(rows, columns)</code>. <code>index_col=0</code> tells pandas the first column is just a row label (here an export index), so it becomes the row <i>index</i> instead of a data column — which is why the column count is one less than the raw file's 226.</div>

In [5]:
# We were asked to "print the columns" — but there are hundreds, so print the count
# first, then the ~25 structured columns, then confirm how many chief-complaint flags
# there are (those we list by body system in section 5, never by hand).
print("Total columns:", df.shape[1], "\n")

structured = [c for c in df.columns if not c.startswith("cc_")]
cc_flags   = [c for c in df.columns if c.startswith("cc_")]

print(f"Structured columns ({len(structured)}):")
for c in structured:
    print("   ", c)
print(f"\nChief-complaint flags (cc_*): {len(cc_flags)}  ->  grouped by body system in section 5")

Total columns: 225 

Structured columns (25):
    dep_name
    esi
    age
    gender
    ethnicity
    race
    lang
    religion
    maritalstatus
    employstatus
    insurance_status
    disposition
    arrivalmode
    arrivalmonth
    arrivalday
    arrivalhour_bin
    previousdispo
    triage_vital_hr
    triage_vital_sbp
    triage_vital_dbp
    triage_vital_rr
    triage_vital_o2
    triage_vital_o2_device
    triage_vital_temp
    triage_glucose

Chief-complaint flags (cc_*): 200  ->  grouped by body system in section 5


<div style="border-left:4px solid #0A2540;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#0A2540">🎓 TUTOR NOTE</b><br>Land the headline live — of ~225 columns, <b>~200 are <code>cc_</code> chief-complaint flags</b>. The dataset is <i>wide and sparse</i>, not deep. Say it out loud; it reframes everything that follows. ~3 min.</div>

In [6]:
# "What TYPE is each column?" — the first question of literacy.
print("Column types at a glance:")
print(df.dtypes.value_counts(), "\n")     # how many numeric vs text columns

# A compact structural summary of just the structured columns:
df[structured].info()

# ...and the first 3 encounters across a few readable columns:
df[["esi", "age", "gender", "race", "triage_vital_hr", "triage_vital_sbp", "disposition"]].head(3)

Column types at a glance:
float64    210
object      15
Name: count, dtype: int64 

<class 'pandas.core.frame.DataFrame'>
Index: 55121 entries, 7 to 433332
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   dep_name                55121 non-null  object 
 1   esi                     55121 non-null  float64
 2   age                     55121 non-null  float64
 3   gender                  55121 non-null  object 
 4   ethnicity               55121 non-null  object 
 5   race                    55121 non-null  object 
 6   lang                    55121 non-null  object 
 7   religion                55121 non-null  object 
 8   maritalstatus           55121 non-null  object 
 9   employstatus            55121 non-null  object 
 10  insurance_status        55121 non-null  object 
 11  disposition             55121 non-null  object 
 12  arrivalmode             55121 non-null  object 
 13  arrivalmont

,esi,age,gender,race,triage_vital_hr,triage_vital_sbp,disposition
7,4.0,87.0,Female,Other,88.0,155.0,Discharge
17,2.0,53.0,Male,Other,118.0,105.0,Admit
40,2.0,49.0,Female,White or Caucasian,76.0,116.0,Discharge


<div style="border-left:4px solid #6C5CE7;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#6C5CE7">🧩 PANDAS IN PLAIN ENGLISH</b><br><code>df.dtypes</code> is the type of each column: <code>object</code> holds text like "Female" or "Admit"; <code>float64</code> holds numbers like a heart rate. <code>.info()</code> lists every column with its type and its count of non-empty values — a fast way to see structure and to spot missing data at a glance.</div>

## 3 · Patient identifiers & outcomes

Every **row is one ED encounter** — one patient's arrival. Before reading any values, notice what is *missing by design*: there is **no name, MRN, date of birth, or address**. Those direct identifiers were stripped during de-identification; the only "ID" in the raw file was an export row-number, which we dropped on load. That is a feature, not a bug — it is what makes the data safe to work with.

What remains in the identifier & outcome family:

- **`age`** — years. In this extract, **adults only (≥ 18)** — no paediatric patients. A core risk modifier: the very old tolerate less.
- **`gender`** — recorded Male/Female here. Clinically relevant (cardiac presentations differ, for example) and a fairness axis.
- **`ethnicity`, `race`** — self-identified categories. **Fairness-sensitive**: these are exactly the axes along which a triage model can become inequitable, so we name them now and watch them all week.
- **`disposition`** — what happened *to* the visit (Admit / Discharge). This is an **outcome**, decided *after* triage — not our triage target, and never a model input. Section 6 treats it properly.
- **`esi` (Triage_Level)** — the triage acuity level assigned at the door. **This is our target** (sections 6–7).

In [7]:
# Age is numeric — summarise it. The others are categories — count their values.
print("AGE (years):")
print(df["age"].describe().round(1), "\n")

for col in ["gender", "race", "ethnicity", "disposition"]:
    print(f"{col}:")
    print(df[col].value_counts(dropna=False), "\n")

AGE (years):
count    55121.0
mean        55.3
std         19.5
min         18.0
25%         40.0
50%         55.0
75%         70.0
max        107.0
Name: age, dtype: float64 

gender:
gender
Female    31744
Male      23377
Name: count, dtype: int64 

race:
race
White or Caucasian                           29435
Black or African American                    15963
Other                                         9016
Patient Refused                                370
Asian                                          175
Unknown                                         76
American Indian or Alaska Native                66
Native Hawaiian or Other Pacific Islander       20
Name: count, dtype: int64 

ethnicity:
ethnicity
Non-Hispanic          45142
Hispanic or Latino     9888
Patient Refused          56
Unknown                  35
Name: count, dtype: int64 

disposition:
disposition
Discharge    34565
Admit        20556
Name: count, dtype: int64 



<div style="border-left:4px solid #6C5CE7;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#6C5CE7">🧩 PANDAS IN PLAIN ENGLISH</b><br>For a text column, <code>.value_counts()</code> tallies how many rows carry each value — your first look at <i>what categories exist</i>. <code>dropna=False</code> makes it also count blanks (NaN) so missing data can't hide. For a number column, <code>.describe()</code> gives count, mean, min/max and quartiles instead.</div>

<div style="border-left:4px solid #0A2540;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#0A2540">🎓 TUTOR NOTE</b><br>Two things to surface. <b>(1) Fairness</b> — <code>race</code>, <code>ethnicity</code>, <code>insurance_status</code> are sensitive; a model can encode inequity through them. <b>(2) Scope</b> — the minimum age is 18, so this is an <i>adult</i> ED extract; a Caribbean deployment covering children would need different data. Note also there are <b>zero missing values</b> here — the source was pre-cleaned/imputed — so don't teach "handle the missingness" today; real data won't be this tidy (that's T2/T3). ~4 min.</div>

## 4 · Vital signs on arrival

The **triage vitals** are the numbers a nurse measures the moment a patient reaches the desk — the body's headline readout, and the strongest *objective* signal feeding the triage decision.

| Column | Vital | Normal adult | Why it matters |
|---|---|---|---|
| `triage_vital_hr`  | Heart rate (bpm)              | 60–100    | > 100 tachycardia; a stress/shock signal |
| `triage_vital_sbp` | Systolic BP (mmHg)            | 90–140    | < 90 → possible shock |
| `triage_vital_dbp` | Diastolic BP (mmHg)           | 60–90     | read together with SBP |
| `triage_vital_rr`  | Respiratory rate (/min)       | 12–20     | the **earliest** sign of deterioration |
| `triage_vital_o2`  | Oxygen saturation (%)         | 95–100    | < 92 concerning |
| `triage_vital_temp`| Temperature (**°F**)          | 97.0–99.5 | **Fahrenheit!** fever ≥ 100.4 °F (38 °C) |
| `triage_glucose`   | Point-of-care glucose (mg/dL) | 70–140    | very high *or* low both flag |

Also here: **`triage_vital_o2_device`** — coded 0/1, whether the patient was on a supplemental-oxygen device when SpO₂ was measured (98% on room air is not the same as 98% on oxygen).

In [8]:
# Clinical reference ranges (general adult triage). Temperature is FAHRENHEIT here.
NORMAL_RANGES = {
    "triage_vital_hr":   (60, 100, "bpm"),
    "triage_vital_sbp":  (90, 140, "mmHg"),
    "triage_vital_dbp":  (60,  90, "mmHg"),
    "triage_vital_rr":   (12,  20, "/min"),
    "triage_vital_o2":   (95, 100, "%"),
    "triage_vital_temp": (97.0, 99.5, "F"),   # NOT Celsius
    "triage_glucose":    (70, 140, "mg/dL"),
}
VITALS = list(NORMAL_RANGES)

# Numeric summary only — no plots today. Compare min/max against the ranges above.
print(df[VITALS].describe().round(1))

print("\nOn a supplemental-O2 device?  (0 = room air, 1 = device)")
print(df["triage_vital_o2_device"].value_counts(dropna=False))

       triage_vital_hr  triage_vital_sbp  triage_vital_dbp  triage_vital_rr  triage_vital_o2  triage_vital_temp  triage_glucose
count          55121.0           55121.0           55121.0          55121.0          55121.0            55121.0         55121.0
mean              86.4             133.7              79.5             17.8             97.0               98.1           130.1
std               17.0              22.6              14.6              2.1              2.1                0.8            73.7
min               32.0              53.0              27.0              8.0             62.0               91.5            16.0
25%               74.0             118.0              70.0             16.0             96.0               97.7            93.0
50%               85.0             132.0              79.0             18.0             98.0               98.0           107.0
75%               97.0             147.0              89.0             18.0             98.0            

<div style="border-left:4px solid #6C5CE7;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#6C5CE7">🧩 PANDAS IN PLAIN ENGLISH</b><br><code>.describe()</code> on numeric columns returns count, mean, std, min, the 25/50/75% percentiles, and max. The 50% row is the <b>median</b>. Comparing min/max against the normal ranges above is how you sanity-check a vital <i>without plotting anything</i>.</div>

<div style="border-left:4px solid #0A2540;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#0A2540">🎓 TUTOR NOTE</b><br>Three points must land. <b>(1)</b> Respiratory rate is the earliest deterioration signal and the most under-recorded vital in real EDs — worth dwelling on. <b>(2)</b> Temperature is <b>Fahrenheit</b> (~98.6 normal); a student who assumes Celsius will call every patient hypothermic. <b>(3)</b> This extract is complete and clipped to plausible values — real triage data is messy; profiling that is T2. Keep it to numbers; resist the urge to plot. ~5 min.</div>

## 5 · Chief complaints — 200 flags, grouped by body system

A **chief complaint** is the patient's stated reason for coming in — "chest pain", "fall", "shortness of breath". The dataset records these as **~200 binary flags** (`cc_*`): a column is **1** if that complaint was recorded for the patient, **0** otherwise. This is a **one-hot** encoding, and it is **sparse** — any patient trips only one or a few flags, so almost every cell is 0.

You cannot read 200 columns one by one, and you should not try. The literacy move is to **group them into body systems** — the way a clinician already thinks — and reason about the *family*, not the flag. Below we sort all 200 into ~12 clinical categories.

In [9]:
cc_flags = [c for c in df.columns if c.startswith("cc_")]

# Ordered (category -> keywords) rules. FIRST MATCH WINS, so specific systems come
# before the Trauma/MSK "pain/injury/swelling" near-catch-all, and Other is last.
CC_CATEGORIES = [
    ("Cardiovascular",             ["chestpain","chesttightness","palpitation","irregularheart","rapidheart","tachycard","hypertens","hypotens","cardiacarrest"]),
    ("Genitourinary/Renal",        ["dysuria","hematuria","urinary","flank","testicle","maleguproblem","femaleguproblem","groin","pelvic","vaginal","breastpain","std"]),
    ("Respiratory",                ["breathing","shortnessofbreath","dyspnea","cough","wheez","respiratory","asthma","hemoptysis","coldlike","nasalcongest","influenza","sinus","sorethroat","uri"]),
    ("Neurological",               ["headache","migraine","dizz","syncope","seizure","stroke","alteredmental","confusion","numbness","lossofconsciousness","unresponsive","lethargy","neurologic","hallucinat","extremityweakness"]),
    ("Gastrointestinal",           ["abdominal","epigastric","nausea","emesis","vomit","diarrhea","constipation","gibleeding","giproblem","rectal","swallowedforeign","ingestion","dehydration"]),
    ("Psych/Behavioral/Substance", ["anxiety","depress","agitation","suicid","homicid","psychot","psychiatric","panic","alcohol","drugproblem","drug/alcohol","addiction","detox","overdose","withdrawal","poison"]),
    ("ENT/Eye/Dental",             ["earpain","earproblem","otalgia","eye","conjunctiv","foreignbodyineye","blurredvision","dental","jaw","epistaxis","facial","oralswelling"]),
    ("Skin/Soft-tissue",           ["rash","cellulitis","abscess","cyst","skin","wound","mass","allergicreaction","edema","bruising"]),
    ("Endocrine/Metabolic/Heme",   ["bloodsugar","glycem","glucose","sicklecell"]),
    ("Constitutional/General",     ["fever","chills","fatigue","bodyache","generalizedbody","weakness"]),
    ("Trauma/Injury/MSK",          ["injury","fall","trauma","laceration","fracture","back","neck","joint","rib","assault","burn","animalbite","insectbite","pain","swelling"]),
]

def categorize(flag):
    stem = flag[3:]                     # drop the "cc_" prefix
    if "crash" in stem:                 # 'crash' contains 'rash' -> disambiguate FIRST
        return "Trauma/Injury/MSK"
    for category, keywords in CC_CATEGORIES:
        if any(k in stem for k in keywords):
            return category
    return "Other/Procedural/Admin"

# Build category -> list of flags, and confirm we placed every one.
from collections import defaultdict
groups = defaultdict(list)
for flag in cc_flags:
    groups[categorize(flag)].append(flag)

order = [name for name, _ in CC_CATEGORIES] + ["Other/Procedural/Admin"]
for name in order:
    print(f"{name:28s} {len(groups[name]):3d} flags")
print("-" * 40)
print(f"{'TOTAL placed':28s} {sum(len(v) for v in groups.values()):3d} of {len(cc_flags)}")

Cardiovascular                 9 flags
Genitourinary/Renal           17 flags
Respiratory                   15 flags
Neurological                  21 flags
Gastrointestinal              16 flags
Psych/Behavioral/Substance    18 flags
ENT/Eye/Dental                18 flags
Skin/Soft-tissue              14 flags
Endocrine/Metabolic/Heme       5 flags
Constitutional/General         8 flags
Trauma/Injury/MSK             49 flags
Other/Procedural/Admin        10 flags
----------------------------------------
TOTAL placed                 200 of 200


<div style="border-left:4px solid #6C5CE7;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#6C5CE7">🧩 PANDAS IN PLAIN ENGLISH</b><br><code>flag.startswith("cc_")</code> is a plain string test — True for every chief-complaint column. Watch the <b>substring trap</b>: <code>"crash"</code> contains <code>"rash"</code>, so a naive rule would file a car crash under skin rash. We check <code>"crash"</code> first to prevent it — a small reminder that keyword matching needs care.</div>

<div style="border-left:4px solid #F4B942;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#F4B942">✏️ YOUR TASK</b><br>Each <code>cc_</code> column is 0/1, so <b>summing a column counts how many patients had that complaint</b>. In the next cell, (1) compute a <b>total per body system</b> — how many complaint-flags fired across all patients in each category — and (2) print the single most common individual complaint. The steps and function names are given; you fill in the code.</div>

In [10]:
# (1) Totals per body system: sum every flag in the category, across all patients.
print("Complaint volume by body system (flag-hits across all encounters):")
system_totals = {name: int(df[flags].sum().sum()) for name, flags in groups.items()}
for name in sorted(system_totals, key=system_totals.get, reverse=True):
    print(f"   {name:28s} {system_totals[name]:>7,}")

# (2) The most common individual complaints overall.
per_flag = df[cc_flags].sum().sort_values(ascending=False)
print("\nTop 5 individual complaints:")
print(per_flag.head(5))
print("\nComplaint flags that never fire in this sample:", int((per_flag == 0).sum()))

Complaint volume by body system (flag-hits across all encounters):
   Trauma/Injury/MSK             13,094
   Gastrointestinal              10,542
   Respiratory                    6,693
   Other/Procedural/Admin         6,551
   Neurological                   5,957
   Cardiovascular                 4,912
   Psych/Behavioral/Substance     4,189
   Genitourinary/Renal            3,418
   Constitutional/General         3,032
   Skin/Soft-tissue               2,464
   ENT/Eye/Dental                 1,571
   Endocrine/Metabolic/Heme         875

Top 5 individual complaints:
cc_abdominalpain        6717.0
cc_other                4491.0
cc_chestpain            3712.0
cc_shortnessofbreath    3098.0
cc_backpain             1997.0
dtype: float64

Complaint flags that never fire in this sample: 1


<div style="border-left:4px solid #6C5CE7;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#6C5CE7">🧩 PANDAS IN PLAIN ENGLISH</b><br>On a 0/1 column, <code>.sum()</code> is just a count of the 1s — "how many patients had this". <code>df[flags].sum()</code> gives one total per column; calling <code>.sum()</code> again collapses those into a single number for the whole system. <code>.sort_values(ascending=False)</code> puts the most common first.</div>

<div style="border-left:4px solid #0A2540;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#0A2540">🎓 TUTOR NOTE</b><br>The trap this section defuses: nobody documents 200 flags by hand. The whole chief-complaint family is <b>one idea</b> — a sparse one-hot block you read by prevalence and by system. If a student starts reading flags individually, pause and re-ask: <i>"what's the body system, and what's the one number we care about?"</i> Trauma/MSK and cardio/respiratory dominate volume — a good hook into which systems drive acuity. ~5 min.</div>

## 6 · Why Triage_Level (`esi`) is the target

Everything so far — who the patient is, their vitals, their complaint — is an **input** the triage nurse can see *at the door*. The **decision** they make from it is a single acuity rating: the **Emergency Severity Index (ESI)**, stored in `esi`. That rating is `Triage_Level`, and it is what we want a model to help assign — so **`esi` is our target**.

Keep two columns straight:

- **`esi` — the target.** Assigned *at* triage, from information available *at* triage. Predicting it means supporting a decision at the moment it is actually made.
- **`disposition` — an outcome, not the target.** Admit / Discharge is decided *hours later*, after tests and treatment. It is never available at triage, so it can never be a model *input* for a triage model — using it would be **leakage** (letting the future leak into the present).

A clean target matters: below, `esi` has a value for every encounter and lives entirely on the 1–5 scale — no missing labels to wrestle with.

In [11]:
counts = df["esi"].value_counts(dropna=False).sort_index()
pct    = (counts / len(df) * 100).round(1)
print("ESI (Triage_Level) distribution:")
for level in counts.index:
    print(f"   ESI {level:>3}   {counts[level]:>7,}   ({pct[level]:>4}%)")
print("\nMissing labels:", int(df["esi"].isna().sum()))

ESI (Triage_Level) distribution:
   ESI 1.0        77   ( 0.1%)
   ESI 2.0    17,924   (32.5%)
   ESI 3.0    27,010   (49.0%)
   ESI 4.0     8,896   (16.1%)
   ESI 5.0     1,214   ( 2.2%)

Missing labels: 0


<div style="border-left:4px solid #0A2540;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#0A2540">🎓 TUTOR NOTE</b><br>The disposition-vs-esi distinction is the conceptual core of the week — spend time here. For your own context: in the original Yale study (Hong et al., 2018) the prediction target was actually <b>admission</b> (<code>disposition</code>); this programme reframes the task around the <b>triage level</b> (<code>esi</code>), and disposition returns as the modelling target in a later week. Either way the leakage lesson holds: never feed a triage model something knowable only <i>after</i> triage. Expect "why not just predict admission?" — that's the right question, and the answer is <i>different decision, different moment</i>. ~4 min.</div>

## 7 · What is ESI? The five levels

The **Emergency Severity Index** is a five-level triage algorithm used across US emergency departments. A nurse assigns a level in seconds by asking, in order:

1. **Is a life-saving intervention needed *now*?** → **ESI 1 (Resuscitation)** — e.g. cardiac arrest, unresponsive.
2. **Is this high-risk, or in severe distress/pain, or is a vital sign in the danger zone?** → **ESI 2 (Emergent)** — e.g. chest pain with risk factors, stroke signs.
3. Otherwise, **how many resources** (labs, imaging, procedures) will this take?
   - Many → **ESI 3 (Urgent)**  ·  One → **ESI 4 (Less urgent)**  ·  None → **ESI 5 (Non-urgent)**.

| Level | Name | Meaning | Typical |
|---|---|---|---|
| **1** | Resuscitation | Immediate life-saving intervention | Cardiac arrest, major trauma |
| **2** | Emergent      | High-risk / can't-wait / danger-zone vitals | Chest pain, stroke alert, sepsis |
| **3** | Urgent        | Stable but needs many resources | Abdominal-pain workup |
| **4** | Less urgent   | One resource | Simple laceration |
| **5** | Non-urgent    | No resources | Medication refill |

**Lower number = sicker.** The counts you printed are dominated by ESI 2–3 with very few ESI 1 — realistic for a hospital ED. And the scale is **ordinal**: the gap from 1→2 is not the same "distance" as 4→5, which matters when we model it later.

In [12]:
labels = {1: "Resuscitation", 2: "Emergent", 3: "Urgent", 4: "Less urgent", 5: "Non-urgent"}
counts = df["esi"].value_counts().sort_index()
print("ESI  ·  name           ·  encounters")
for level, n in counts.items():
    print(f"   {int(level)}  ·  {labels[int(level)]:<13}  ·  {n:>7,}")

ESI  ·  name           ·  encounters
   1  ·  Resuscitation  ·       77
   2  ·  Emergent       ·   17,924
   3  ·  Urgent         ·   27,010
   4  ·  Less urgent    ·    8,896
   5  ·  Non-urgent     ·    1,214


<div style="border-left:4px solid #0A2540;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#0A2540">🎓 TUTOR NOTE</b><br>Run the 1→2→3 logic as a live Q&A — give a mini-vignette and have the room call the level ("72-year-old, chest pain, HR 120" → 2; "med refill" → 5). The subtle one is <b>ESI 2 vs 3</b>: a danger-zone vital up-triages an otherwise-3 to a 2 — which is exactly why vitals earned their own section. Don't belabour the resource-counting edge cases. ~5 min.</div>

## 8 · Q&A & discussion prompts

Use these to check understanding and open discussion — no coding required:

1. **Units.** Convert one normal temperature bound from °F to °C. What would break in a model if the two units were mixed in one column?
2. **Fairness.** Name the three columns most likely to introduce bias into a triage model, and say through what mechanism.
3. **Leakage.** A teammate wants to add `disposition` as an input "because it's so predictive." What do you tell them?
4. **Chief complaints.** From the prevalence you computed, name two complaint categories you'd expect to skew toward high acuity (ESI 1–2), and two toward low (ESI 4–5).
5. **Transfer.** Which parts of this US dataset would you most distrust when planning a Caribbean ED deployment, and why?

<div style="border-left:4px solid #0A2540;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#0A2540">🎓 TUTOR NOTE</b><br>Answer sketches. <b>1)</b> °C = (°F − 32) × 5/9, so 100.4 °F = 38 °C; mixed units make one "fever" look normal and corrupt every threshold. <b>2)</b> <code>race</code>, <code>ethnicity</code>, <code>insurance_status</code> (proxy for access). <b>3)</b> It's an outcome known only after triage — leakage; predictive in training, useless and misleading at deployment. <b>4)</b> High: cardiovascular, neurological (stroke), respiratory distress; Low: ENT/dental, minor MSK, medication refill. <b>5)</b> Insurance/access fields, US-specific arrival modes, and any case-mix differences (disease prevalence, demographics) that won't match a Caribbean ED.</div>

## 9 · Wrap-up

You can now read this dataset the way a clinician reads a patient — **by category, not column-by-column**. You can name the feature families, read the vitals (in Fahrenheit), see the 200 chief-complaint flags as ~12 body systems, and explain why `esi` is the target and what its five levels mean.

**Next — Tutorial 2: Profiling.** We stop reading and start measuring: data types, missingness, distributions, and outliers — the first real test of whether this data is good enough to model.

<div style="border-left:4px solid #0A2540;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#0A2540">🎓 TUTOR NOTE</b><br><b>FACILITATION (~30–40 min):</b> 3 intro · 4 load + columns + types · 5 identifiers/outcomes · 6 vitals · 8 chief-complaint grouping + prevalence · 5 why-esi-is-target · 5 ESI levels · remainder Q&A. <b>Three things every student must leave with:</b> (1) temperature is Fahrenheit; (2) ~200 columns are sparse <code>cc_</code> flags read by body system; (3) <code>disposition</code> is an outcome, <code>esi</code> is the target. <b>Discord:</b> "Post your °F→°C conversion and the one column you'd most distrust for a Caribbean ED."</div>